# 🎯 Notebook 04 — Ranking Experiments

This notebook explores the **multi-objective ranking formula** and experiments with different scenarios.

## Ranking Formula
```
RankScore(i) = α × P(click_i | cart)
             + β × P(click_i) × Margin(i)
             + γ × PriceFit(i, cart_total)
             + δ × Diversity(i, already_shown)
             - λ × max(0, KPT_i - KPT_cart)
```
Weights: α=0.35, β=0.30, γ=0.20, δ=0.10, λ=0.05

## Experiments
1. Biryani add-on ranking (sunny vs rainy weather)
2. Late-night scenario (Maggi + KPT filter effect)
3. Group order scenario
4. Weight sensitivity analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded')

In [ ]:
# Define the ranking engine
ALPHA, BETA, GAMMA, DELTA, LAMBDA = 0.35, 0.30, 0.20, 0.10, 0.05

def price_fit_score(addon_price, cart_total, target_ratio=0.17, sigma=0.1):
    """Gaussian around 17% of cart total."""
    ratio = addon_price / max(cart_total, 1)
    return np.exp(-0.5 * ((ratio - target_ratio) / sigma) ** 2)

def rank_addons(candidates, cart_total, cart_max_kpt, shown_categories=None, weights=None):
    """Rank add-on candidates using the SmartCart multi-objective formula."""
    if shown_categories is None:
        shown_categories = set()
    a, b, g, d, l = (weights or (ALPHA, BETA, GAMMA, DELTA, LAMBDA))
    scored = []
    for item in candidates:
        p_click = item['p_click']
        margin  = item['margin']
        price   = item['price']
        kpt     = item['kpt']
        cat     = item.get('category', 'Unknown')

        pf   = price_fit_score(price, cart_total)
        div  = 1.0 if cat not in shown_categories else 0.0
        kpt_penalty = max(0, kpt - cart_max_kpt)

        score = (a * p_click
                 + b * p_click * margin
                 + g * pf
                 + d * div
                 - l * kpt_penalty)

        scored.append({**item, 'score': round(score, 4),
                       'kpt_penalty': kpt_penalty,
                       'price_fit': round(pf, 3),
                       'diversity_bonus': div})
        shown_categories.add(cat)

    return sorted(scored, key=lambda x: x['score'], reverse=True)

def kpt_filter(candidates, cart_max_kpt):
    """Zero-ETA-Impact: block add-ons with KPT > cart max KPT."""
    approved, blocked = [], []
    for item in candidates:
        if item['kpt'] <= cart_max_kpt:
            approved.append(item)
        else:
            delay = item['kpt'] - cart_max_kpt
            blocked.append({**item, 'block_reason': f"KPT {item['kpt']}min > cart max {cart_max_kpt}min (+{delay}min delay)"})
    return approved, blocked

print('Ranking engine defined')

In [ ]:
# Experiment 1: Biryani add-ons — sunny vs rainy weather
biryani_candidates_base = [
    {'name': 'Raita',          'price': 49,  'kpt': 3,  'margin': 0.65, 'category': 'Sides',    'p_click': 0.72},
    {'name': 'Coke 330ml',     'price': 59,  'kpt': 1,  'margin': 0.55, 'category': 'Beverages','p_click': 0.60},
    {'name': 'Gulab Jamun',    'price': 69,  'kpt': 5,  'margin': 0.58, 'category': 'Desserts', 'p_click': 0.50},
    {'name': 'Masala Chai',    'price': 39,  'kpt': 3,  'margin': 0.70, 'category': 'Beverages','p_click': 0.35},
    {'name': 'Hot Chocolate',  'price': 79,  'kpt': 5,  'margin': 0.52, 'category': 'Beverages','p_click': 0.30},
    {'name': 'Garlic Naan',    'price': 49,  'kpt': 8,  'margin': 0.60, 'category': 'Bread',    'p_click': 0.45},
    {'name': 'Shorba',         'price': 79,  'kpt': 6,  'margin': 0.55, 'category': 'Soups',    'p_click': 0.38},
]

# Weather boosts
def apply_weather(candidates, weather):
    """Apply weather-based probability boosts."""
    boosted = []
    for c in candidates:
        item = dict(c)
        if weather == 'rainy':
            if item['category'] in ['Beverages'] and item['name'] in ['Masala Chai', 'Hot Chocolate', 'Shorba']:
                item['p_click'] = min(1.0, item['p_click'] * 1.45)
            elif item['name'] == 'Coke 330ml':
                item['p_click'] *= 0.6
        elif weather == 'sunny':
            if item['name'] == 'Coke 330ml':
                item['p_click'] = min(1.0, item['p_click'] * 1.35)
        boosted.append(item)
    return boosted

cart_total = 280
cart_max_kpt = 20

ranked_sunny = rank_addons(apply_weather(biryani_candidates_base, 'sunny'), cart_total, cart_max_kpt)
ranked_rainy = rank_addons(apply_weather(biryani_candidates_base, 'rainy'), cart_total, cart_max_kpt)

df_sunny = pd.DataFrame(ranked_sunny)[['name', 'score', 'p_click', 'price_fit']]
df_rainy = pd.DataFrame(ranked_rainy)[['name', 'score', 'p_click', 'price_fit']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, df, title, color in zip(axes,
                                 [df_sunny, df_rainy],
                                 ['☀️ Sunny Weather — Biryani Add-ons', '🌧️ Rainy Weather — Biryani Add-ons'],
                                 ['#F39C12', '#3498DB']):
    bars = ax.barh(df['name'][::-1], df['score'][::-1], color=color, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Rank Score')
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.003, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=9)

plt.suptitle('Weather Context Changes Rankings', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nTop 3 (Sunny):', [r['name'] for r in ranked_sunny[:3]])
print('Top 3 (Rainy):', [r['name'] for r in ranked_rainy[:3]])

In [ ]:
# Experiment 2: KPT Filter — Maggi (late night)
maggi_candidates = [
    {'name': 'Masala Chai',    'price': 39,  'kpt': 3,  'margin': 0.70, 'category': 'Beverages', 'p_click': 0.80},
    {'name': 'Packaged Chips', 'price': 20,  'kpt': 1,  'margin': 0.72, 'category': 'Snacks',    'p_click': 0.65},
    {'name': 'Garlic Bread',   'price': 99,  'kpt': 15, 'margin': 0.55, 'category': 'Bread',     'p_click': 0.55},
    {'name': 'Hot Chocolate',  'price': 79,  'kpt': 5,  'margin': 0.52, 'category': 'Beverages', 'p_click': 0.50},
    {'name': 'Paneer Tikka',   'price': 149, 'kpt': 20, 'margin': 0.48, 'category': 'Starters',  'p_click': 0.42},
    {'name': 'Boiled Egg',     'price': 29,  'kpt': 4,  'margin': 0.65, 'category': 'Eggs',      'p_click': 0.55},
]

MAGGI_CART_TOTAL = 59
MAGGI_KPT = 5  # Maggi takes 5 min

approved, blocked = kpt_filter(maggi_candidates, MAGGI_KPT)
ranked_approved = rank_addons(approved, MAGGI_CART_TOTAL, MAGGI_KPT)

print('🍜 Cart: Maggi (KPT=5min, Cart Total=₹59)')
print('=' * 60)
print('\n✅ APPROVED (KPT ≤ 5 min):')
for i, r in enumerate(ranked_approved, 1):
    print(f"  {i}. {r['name']:<20} ₹{r['price']:>3}  KPT={r['kpt']}min  score={r['score']:.3f}")

print(f'\n❌ BLOCKED BY KPT FILTER ({len(blocked)} items):')
for b in blocked:
    print(f"  ✗ {b['name']:<20} KPT={b['kpt']}min → {b['block_reason']}")

# Visualize
all_items = [{'name': r['name'], 'score': r['score'], 'status': 'Approved', 'kpt': r['kpt']} for r in ranked_approved] + \
            [{'name': b['name'], 'score': 0, 'status': 'Blocked (KPT)', 'kpt': b['kpt']} for b in blocked]
df_all = pd.DataFrame(all_items).sort_values('score', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ECC71' if s == 'Approved' else '#E74C3C' for s in df_all['status']]
bars = ax.barh(df_all['name'], df_all['score'], color=colors)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_title('🔑 KPT Filter Effect — Maggi (late night)\nGreen=Approved, Red=Blocked', fontsize=13, fontweight='bold')
ax.set_xlabel('Rank Score (0 = Blocked)')
for bar, row in zip(bars, df_all.itertuples()):
    kpt_label = f'KPT={row.kpt}min'
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            kpt_label, va='center', fontsize=9)

green_patch = mpatches.Patch(color='#2ECC71', label='Approved (KPT ≤ 5min)')
red_patch   = mpatches.Patch(color='#E74C3C', label='Blocked  (KPT > 5min)')
ax.legend(handles=[green_patch, red_patch])
plt.tight_layout()
plt.show()

In [ ]:
# Experiment 3: Weight sensitivity analysis
candidates_base = [
    {'name': 'Coke 330ml',   'price': 59,  'kpt': 1,  'margin': 0.55, 'category': 'Beverages', 'p_click': 0.75},
    {'name': 'Fries',         'price': 89,  'kpt': 5,  'margin': 0.68, 'category': 'Sides',     'p_click': 0.50},
    {'name': 'Brownie',       'price': 99,  'kpt': 3,  'margin': 0.62, 'category': 'Desserts',  'p_click': 0.45},
    {'name': 'Mineral Water', 'price': 25,  'kpt': 1,  'margin': 0.75, 'category': 'Beverages', 'p_click': 0.55},
    {'name': 'Garlic Bread',  'price': 99,  'kpt': 10, 'margin': 0.58, 'category': 'Bread',     'p_click': 0.40},
]

cart_total = 350
cart_max_kpt = 20

weight_configs = {
    'Default (α=0.35 β=0.30)': (0.35, 0.30, 0.20, 0.10, 0.05),
    'Revenue-heavy (β=0.50)':  (0.20, 0.50, 0.15, 0.10, 0.05),
    'Conversion-only (α=0.80)': (0.80, 0.05, 0.05, 0.05, 0.05),
    'Price-fit-heavy (γ=0.50)': (0.20, 0.15, 0.50, 0.10, 0.05),
}

results = {}
for config_name, weights in weight_configs.items():
    ranked = rank_addons(candidates_base, cart_total, cart_max_kpt, weights=weights)
    results[config_name] = [r['name'] for r in ranked]

print('Ranking sensitivity to weight configurations:')
print(f'{"Config":<35} {"#1":<18} {"#2":<18} {"#3"}')
print('-' * 75)
for cfg, ranking in results.items():
    print(f'{cfg:<35} {ranking[0]:<18} {ranking[1]:<18} {ranking[2]}')

# Heatmap of scores across configs
score_matrix = []
for config_name, weights in weight_configs.items():
    ranked = rank_addons(candidates_base, cart_total, cart_max_kpt, weights=weights)
    score_row = {r['name']: r['score'] for r in ranked}
    score_matrix.append(score_row)

df_heat = pd.DataFrame(score_matrix, index=list(weight_configs.keys()))

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(df_heat.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(df_heat.columns)))
ax.set_xticklabels(df_heat.columns, rotation=30, ha='right')
ax.set_yticks(range(len(df_heat.index)))
ax.set_yticklabels(df_heat.index)
for i in range(len(df_heat.index)):
    for j in range(len(df_heat.columns)):
        ax.text(j, i, f'{df_heat.values[i, j]:.3f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Rank Score')
ax.set_title('Ranking Score Heatmap by Weight Configuration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Experiment 4: Group vs Solo order
group_candidates = [
    {'name': 'Coke 2L',         'price': 120, 'kpt': 1,  'margin': 0.50, 'category': 'Beverages', 'p_click': 0.65},
    {'name': 'Large Fries',     'price': 119, 'kpt': 6,  'margin': 0.65, 'category': 'Sides',     'p_click': 0.58},
    {'name': 'Garlic Bread',    'price': 99,  'kpt': 10, 'margin': 0.58, 'category': 'Bread',     'p_click': 0.50},
    {'name': 'Coke 330ml',      'price': 59,  'kpt': 1,  'margin': 0.55, 'category': 'Beverages', 'p_click': 0.60},
    {'name': 'Brownie Platter', 'price': 199, 'kpt': 5,  'margin': 0.55, 'category': 'Desserts',  'p_click': 0.45},
]
solo_candidates = [
    {'name': 'Coke 330ml',   'price': 59,  'kpt': 1,  'margin': 0.55, 'category': 'Beverages', 'p_click': 0.70},
    {'name': 'Small Fries',  'price': 69,  'kpt': 5,  'margin': 0.65, 'category': 'Sides',     'p_click': 0.58},
    {'name': 'Ice Cream',    'price': 79,  'kpt': 2,  'margin': 0.60, 'category': 'Desserts',  'p_click': 0.45},
    {'name': 'Mineral Water','price': 25,  'kpt': 1,  'margin': 0.75, 'category': 'Beverages', 'p_click': 0.55},
]

GROUP_CART = 850
SOLO_CART  = 250

ranked_group = rank_addons(group_candidates, GROUP_CART, 30)[:2]   # Group → 2 recs
ranked_solo  = rank_addons(solo_candidates,  SOLO_CART,  20)[:4]   # Solo  → 4 recs

print('👥 GROUP ORDER (4 items, ₹850 cart) → 2 recommendations:')
for r in ranked_group:
    print(f"  ✅ {r['name']:<20} ₹{r['price']}  score={r['score']:.3f}")

print(f'\n🙋 SOLO ORDER (1 item, ₹250 cart) → 4 recommendations:')
for r in ranked_solo:
    print(f"  ✅ {r['name']:<20} ₹{r['price']}  score={r['score']:.3f}")

print('\n✨ Group orders get fewer, higher-value add-ons; Solo orders get more, budget-friendly add-ons')